# Kacchi Aloo: final-selection portfolio (P01–P22)

This competition lets a team select up to 25 final submissions, and the private leaderboard counts the best of them. Every team uses nearly the same potato-per-guest band, so the private ranking is decided by a few near-edge weddings whose labels are noisy. These tickets cover different plausible outcomes for those weddings. Each is a rule computed from the data:

* **band(lo, hi)**: predict 1 when lo ≤ aloo/guest ≤ hi (shifted edges);
* **window**: the sharp band A with a block of its most uncertain rows flipped. Uncertainty ranks come from the band's out-of-fold error rate at each distance from the edge (the same ranking that produced B).

Which rules to include was chosen by a simulation that maximises the chance of finishing first and in the top 5 (`solution/portfolio25.py`, E23). No test row is labelled by hand.

Run on Kaggle with the competition data attached; it writes `P01.csv` … one file per ticket.

In [1]:
import glob, hashlib, os, unicodedata
import numpy as np, pandas as pd

found = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)) + [p for p in ("../data/train.csv", "data/train.csv") if os.path.exists(p)]
root = os.path.dirname(found[0])
tr_raw = pd.read_csv(f"{root}/train.csv", dtype=str)
te_raw = pd.read_csv(f"{root}/test.csv", dtype=str)


def to_number(v):
    """Float from a cell; any Unicode decimal digit (e.g. Bangla) becomes ASCII."""
    if pd.isna(v):
        return np.nan
    s = "".join(str(unicodedata.decimal(c)) if unicodedata.decimal(c, None) is not None else c
                for c in str(v).strip().replace(",", ""))
    try:
        return float(s)
    except ValueError:
        return np.nan


def add_apg(d):
    """aloo per guest; the few aloo counts typed x10 (ratio > 3) are divided by 10."""
    guests, aloo = d.guests.map(to_number), d.aloo_count.map(to_number)
    aloo = np.where(aloo / guests > 3.0, aloo / 10, aloo)
    return d.assign(apg=aloo / guests)


tr = add_apg(tr_raw).dropna(subset=["apg"]).reset_index(drop=True)   # 24 train rows have no aloo_count
tr["y"] = tr.went_back_for_seconds.astype(int)
te = add_apg(te_raw)
print(f"data: {root} | train {len(tr)} rows with aloo_count | test {len(te)} rows")

data: ../data | train 776 rows with aloo_count | test 400 rows


In [2]:
def midcut_cuts(r, y):
    """Each edge sits at the midpoint between neighbouring training ratios that maximises accuracy on its side of
    1.5 (median of tied optima)."""
    def best_cut(x, yy, side):
        xs = np.sort(np.unique(x))
        c = (xs[:-1] + xs[1:]) / 2
        accs = np.array([((x >= t) == yy).mean() if side == "lo" else ((x <= t) == yy).mean() for t in c])
        b = np.flatnonzero(accs == accs.max())
        return c[b[len(b) // 2]]
    m = r < 1.5
    return best_cut(r[m], y[m], "lo"), best_cut(r[~m], y[~m], "hi")


def band(x, lo, hi):
    return ((x >= lo) & (x <= hi)).astype(int)

In [3]:
from sklearn.model_selection import StratifiedKFold


def sharp_band_probabilities(tr, te, bins=(0.0, 0.01, 0.035, 0.1, 0.4, np.inf), repeats=10, seed=0):
    """P(y=1) per test row = the sharp band's out-of-fold error rate at that distance from the nearest edge.
    Test rows inside the training gap around an edge get P from a uniform prior on where the cut lies."""
    r, y = tr.apg.values, tr.y.values
    lo, hi = midcut_cuts(r, y)
    d_oof, e_oof = [], []
    for k in range(repeats):
        for t, v in StratifiedKFold(5, shuffle=True, random_state=seed + k).split(tr, y):
            l2, h2 = midcut_cuts(r[t], y[t])
            d_oof.append(np.minimum(abs(r[v] - l2), abs(r[v] - h2)))
            e_oof.append(band(r[v], l2, h2) != y[v])
    d_oof, e_oof = np.concatenate(d_oof), np.concatenate(e_oof)
    idx = np.digitize(d_oof, bins[1:-1])
    err = np.array([(e_oof[idx == b].sum() + 0.5) / ((idx == b).sum() + 1.0) for b in range(len(bins) - 1)])
    below_lo, above_lo = r[r < lo].max(), r[(r >= lo) & (r < 1.5)].min()
    below_hi, above_hi = r[(r <= hi) & (r > 1.5)].max(), r[r > hi].min()
    x = te.apg.values
    d = np.minimum(abs(x - lo), abs(x - hi))
    e = err[np.digitize(d, bins[1:-1])]
    P = np.where(band(x, lo, hi) == 1, 1 - e, e)
    q_lo = np.clip((x - below_lo) / (above_lo - below_lo), 0, 1)
    q_hi = np.clip((above_hi - x) / (above_hi - below_hi), 0, 1)
    g_lo, g_hi = (x > below_lo) & (x < above_lo), (x > below_hi) & (x < above_hi)
    P = np.where(g_lo, q_lo * (1 - err[0]) + (1 - q_lo) * err[0], P)
    P = np.where(g_hi, q_hi * (1 - err[0]) + (1 - q_hi) * err[0], P)
    return P, (lo, hi), err


P, (LO, HI), err = sharp_band_probabilities(tr, te)
print(f"cut-points {LO:.5f} / {HI:.5f}; out-of-fold error rate by distance to the edge "
      f"(0-0.01, 0.01-0.035, 0.035-0.1, 0.1-0.4, >0.4): {np.round(err, 3)}")

cut-points 0.97068 / 2.00983; out-of-fold error rate by distance to the edge (0-0.01, 0.01-0.035, 0.035-0.1, 0.1-0.4, >0.4): [0.127 0.374 0.148 0.022 0.   ]


In [4]:
SPECS = [
 {
  "family": "window",
  "side": "upper",
  "start": 2,
  "width": 8
 },
 {
  "family": "band",
  "lo": 1.00608,
  "hi": 2.02353
 },
 {
  "family": "window",
  "side": "upper",
  "start": 7,
  "width": 7
 },
 {
  "family": "band",
  "lo": 1.00689,
  "hi": 1.98867
 },
 {
  "family": "window",
  "side": "lower",
  "start": 5,
  "width": 5
 },
 {
  "family": "window",
  "side": "both",
  "start": 8,
  "width": 6
 },
 {
  "family": "band",
  "lo": 0.95816,
  "hi": 1.9807
 },
 {
  "family": "window",
  "side": "both",
  "start": 19,
  "width": 7
 },
 {
  "family": "band",
  "lo": 0.97052,
  "hi": 2.02641
 },
 {
  "family": "window",
  "side": "lower",
  "start": 2,
  "width": 4
 },
 {
  "family": "window",
  "side": "both",
  "start": 16,
  "width": 4
 },
 {
  "family": "window",
  "side": "upper",
  "start": 0,
  "width": 5
 },
 {
  "family": "window",
  "side": "upper",
  "start": 13,
  "width": 5
 },
 {
  "family": "window",
  "side": "both",
  "start": 12,
  "width": 4
 },
 {
  "family": "window",
  "side": "both",
  "start": 1,
  "width": 5
 },
 {
  "family": "band",
  "lo": 0.9843,
  "hi": 1.97931
 },
 {
  "family": "band",
  "lo": 0.99333,
  "hi": 2.01654
 },
 {
  "family": "window",
  "side": "both",
  "start": 23,
  "width": 5
 },
 {
  "family": "window",
  "side": "both",
  "start": 15,
  "width": 7
 },
 {
  "family": "band",
  "lo": 0.96938,
  "hi": 1.95509
 },
 {
  "family": "band",
  "lo": 1.00347,
  "hi": 1.98148
 },
 {
  "family": "band",
  "lo": 0.96462,
  "hi": 2.01953
 }
]
EXPECTED = {
 "P01.csv": "c53a188e73bd4ebf4765dde713cdf1c9",
 "P02.csv": "3cdbce646338f577f9b7c9f4b2c2c6ef",
 "P03.csv": "112eba3f4c4edb4d86b1138f85b68e9d",
 "P04.csv": "e1970ca0c795ab36d7e4ca3fe1d668b6",
 "P05.csv": "6819a71dc9488cc58ee2784dc9a1a7bf",
 "P06.csv": "7997d2a083e7385d9e1716a6a191a0b2",
 "P07.csv": "b16cd9c9c30f616a664c409e3820831f",
 "P08.csv": "c19945687dd280999c0fce47143f0f5e",
 "P09.csv": "3003326efb3bff3115f4058b0e51487e",
 "P10.csv": "92292f0bd86f850394003f90948efb3e",
 "P11.csv": "26d9d0b6e59e6baed10ea72d8bf001ef",
 "P12.csv": "262d3618f4954135c0cc62c8dd9cef89",
 "P13.csv": "0c9c8c1e1a4220940c2f54963dfb64a7",
 "P14.csv": "e541894349cb6393a1818208876e2ad8",
 "P15.csv": "6154d9cc73569204919874be0beba55d",
 "P16.csv": "06e301fb28c32ee251c2cf105031c9fa",
 "P17.csv": "d82ee3ad2a890e12bd8d2ba8bac644eb",
 "P18.csv": "caf6053ae7d535e68aed7d7d8363fcb3",
 "P19.csv": "5fa4f233c88e4c9d126afc36a8bb3db1",
 "P20.csv": "f8a4e6c01a2d194b10fc1cb7055f211a",
 "P21.csv": "1f64ee8fa1037c7d90aa286dd6644ec6",
 "P22.csv": "b7f07c4244f32d672498a202fbf6d793"
}

A = band(te.apg.values, LO, HI)
order = np.lexsort((np.minimum(abs(te.apg.values - LO), abs(te.apg.values - HI)), np.abs(P - 0.5)))
low = te.apg.values < 1.5
side_order = {"both": order, "lower": order[low[order]], "upper": order[~low[order]]}


def ticket(spec):
    if spec["family"] == "band":
        return band(te.apg.values, spec["lo"], spec["hi"])
    v = A.copy()
    v[side_order[spec["side"]][spec["start"]:spec["start"] + spec["width"]]] ^= 1
    return v


def describe(spec):
    if spec["family"] == "band":
        return f"band {spec['lo']} <= aloo/guest <= {spec['hi']}"
    edge = {"both": "both edges", "lower": "lower edge", "upper": "upper edge"}[spec["side"]]
    return f"A + flip uncertainty ranks {spec['start'] + 1}-{spec['start'] + spec['width']} ({edge})"


rows = []
for k, spec in enumerate(SPECS, 1):
    v = ticket(spec)
    name = f"P{k:02d}.csv"
    sub = pd.DataFrame({"wedding_id": te.wedding_id, "went_back_for_seconds": v.astype(int)})
    sub.to_csv(name, index=False)
    fp = hashlib.md5("".join(f"{i}{p}" for i, p in zip(sub.wedding_id, sub.went_back_for_seconds)).encode()).hexdigest()
    rows.append(dict(file=name, rule=describe(spec), rows_vs_A=int((v != A).sum()), ones=int(v.sum()), matches_plan=fp == EXPECTED[name]))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print("all tickets match the optimizer's files:", bool(summary.matches_plan.all()))

   file                                          rule  rows_vs_A  ones  matches_plan
P01.csv  A + flip uncertainty ranks 3-10 (upper edge)          8   186          True
P02.csv         band 1.00608 <= aloo/guest <= 2.02353         10   182          True
P03.csv  A + flip uncertainty ranks 8-14 (upper edge)          7   181          True
P04.csv         band 1.00689 <= aloo/guest <= 1.98867          8   174          True
P05.csv  A + flip uncertainty ranks 6-10 (lower edge)          5   183          True
P06.csv  A + flip uncertainty ranks 9-14 (both edges)          6   184          True
P07.csv          band 0.95816 <= aloo/guest <= 1.9807          9   181          True
P08.csv A + flip uncertainty ranks 20-26 (both edges)          7   179          True
P09.csv         band 0.97052 <= aloo/guest <= 2.02641          7   189          True
P10.csv   A + flip uncertainty ranks 3-6 (lower edge)          4   182          True
P11.csv A + flip uncertainty ranks 17-20 (both edges)          4 